# Human Action Recognition — Step 1: Data Preprocessing

**Goal of the project:** teach a neural network to look at a photo of a person and say what they are doing
(for example *running*, *eating*, *sleeping*).

**Goal of this notebook:** get the images ready for the neural network. A network can't use raw photos directly:

| Problem with raw data | What we do about it |
|---|---|
| Labels are words ("running") | Turn each label into a number (0–14) |
| Photos have different sizes | Resize them all to the same size |
| Pixel values are 0–255 | Scale them to 0–1 so training is stable |
| We need to check the model fairly | Split off a *validation* set the model never trains on |
| Only ~12,600 photos | *Augmentation*: random flips/zooms so the model sees more variety |

**Dataset:** Kaggle "Human Action Recognition (HAR) Dataset" — 15 actions, ~12,600 labeled training images.

## 1. Import libraries
- **pandas** – read the CSV label file as a table
- **numpy** – maths on arrays of numbers
- **matplotlib** – draw charts and show images
- **tensorflow / keras** – build the image pipeline (and later the neural network)

In [ ]:
import os, glob
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # hide TensorFlow's technical log messages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

## 2. Find the dataset
On Kaggle, attached datasets live under `/kaggle/input/`. This cell searches for the label file
so it works no matter what the folder is called.

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
print("Data folder:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))

## 3. Look at the labels
`Training_set.csv` has two columns: the image **filename** and its **label** (the action).

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
print("Number of labeled images:", len(df))
df.head(10)

## 4. How many images per action?
If one action had far more images than the others, the model could become biased toward it.
This chart checks that the classes are **balanced**.

In [ ]:
counts = df["label"].value_counts()
print("Number of classes:", len(counts))
counts.plot(kind="barh", figsize=(8, 6), title="Images per action")
plt.xlabel("Number of images")
plt.tight_layout()
plt.show()

## 5. Look at some example images
Always look at your data before training — it tells you how hard the task is.

In [ ]:
sample = df.sample(12, random_state=1)
plt.figure(figsize=(12, 9))
for i, (_, row) in enumerate(sample.iterrows()):
    img = Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"]))
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Check image sizes
The photos come in many different sizes, but a neural network needs every input to be the **same size**.
That's why we'll resize them all in a later step.

In [ ]:
sizes = [Image.open(os.path.join(TRAIN_IMG_DIR, f)).size for f in df["filename"].sample(300, random_state=0)]
widths, heights = zip(*sizes)
print(f"Width:  min {min(widths)}, max {max(widths)}, average {np.mean(widths):.0f}")
print(f"Height: min {min(heights)}, max {max(heights)}, average {np.mean(heights):.0f}")

## 7. Turn labels into numbers (label encoding)
Neural networks work with numbers, not words, so each action gets an ID:
`calling → 0`, `clapping → 1`, and so on.

In [ ]:
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)

NUM_CLASSES = len(class_names)
for name, i in label_to_id.items():
    print(i, name)

## 8. Split into training and validation sets
- **Training set (80%)** – the model learns from these.
- **Validation set (20%)** – kept aside to check how well the model does on images it has *never seen*.

Note: the provided `Testing_set.csv` has **no labels**, so we can't use it to measure accuracy.
That's why we make our own validation set from the labeled data.

`stratify` keeps the same mix of actions in both sets.

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label_id"], random_state=42
)
print("Training images:  ", len(train_df))
print("Validation images:", len(val_df))

## 9. Build the image pipeline: load → resize → normalize
For every image we:
1. **Load** the JPEG file from disk.
2. **Resize** it to 224 × 224 pixels (the standard size for most image networks).
3. **Normalize**: divide pixel values by 255 so they are between 0 and 1 instead of 0 and 255.

We also group images into **batches** of 32 — the network learns from 32 images at a time.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)        # 3 channels = red, green, blue
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))   # same size for every image
    img = img / 255.0                                  # scale pixels to 0-1
    return img, label

def make_dataset(frame, shuffle):
    paths = [os.path.join(TRAIN_IMG_DIR, f) for f in frame["filename"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, frame["label_id"].values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=42)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
print("Training batches:  ", len(train_ds), f"(of {BATCH_SIZE} images each)")
print("Validation batches:", len(val_ds))
print("Each image is", IMG_SIZE, "x", IMG_SIZE, "pixels with 3 colour channels (RGB)")

## 10. Data augmentation
We only have ~10,000 training photos. **Augmentation** creates slightly changed copies on the fly
(flipped, rotated, zoomed), so the model learns that "running" is still running if the photo is mirrored.
This helps the model **generalize** instead of memorizing.

Augmentation is only used on the training set, never on validation.

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="augmentation")

# Show one image with 6 different random augmentations
images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(6):
    aug = augment(images[:1], training=True)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(aug.numpy())
    plt.title(class_names[int(labels[0])])
    plt.axis("off")
plt.suptitle("Same photo, different augmentations")
plt.tight_layout()
plt.show()

## 11. Final check
Each batch is a block of numbers with shape `(32, 224, 224, 3)`:
32 images × 224 pixels high × 224 pixels wide × 3 colour channels. This is exactly what the neural network will take as input.

In [ ]:
images, labels = next(iter(train_ds))
print("Batch of images:", images.shape)
print("Batch of labels:", labels.shape)
print("Pixel value range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

## Summary
1. Loaded **~12,600 labeled images** across **15 actions**.
2. Checked the classes are balanced and looked at examples.
3. Turned text labels into numbers.
4. Split into **80% training / 20% validation**.
5. Resized every image to **224 × 224** and scaled pixels to **0–1**.
6. Added **augmentation** (flip, rotate, zoom) for training.

**Next step:** build and train the neural network (a CNN) on `train_ds` and measure accuracy on `val_ds`.